In [ ]:
# === Round-robin ELO arena ===
# v1_new = DeepSeek base + 6 改进 (F1/F2/F3/F4/F7/F8)
# v1_old = 上一版（A 方案调参后的 v1）
# deepseek = 原始 v1_deepseek（最强基线）
# chatgpt = v1_chatgpt（对手采样 minimax 风格）
# nearest = 官方 baseline (+1 sniper)
import sys, importlib, math as _math
HERE = "/data1/huanghao3/KK/Orbit_War/scripts"
if HERE not in sys.path:
    sys.path.insert(0, HERE)

# Force-reload all agent modules so iterative tweaks take effect
import v1.main as _v1_new
importlib.reload(_v1_new)
import v1_old.main as _v1_old
importlib.reload(_v1_old)
import v1_deepseek.main as _v1_deepseek
importlib.reload(_v1_deepseek)
import v1_chatgpt.main as _v1_chatgpt
importlib.reload(_v1_chatgpt)
import eval_arena as _arena_mod
importlib.reload(_arena_mod)

v1_new = _v1_new.agent
v1_old = _v1_old.agent
deepseek_agent = _v1_deepseek.agent
chatgpt_agent = _v1_chatgpt.agent
Arena = _arena_mod.Arena


def nearest_sniper(obs, config=None):
    """README baseline: send target.ships+1 (or >=20) to the nearest non-own planet."""
    from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as _P
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [_P(*p) for p in raw]
    targets = [p for p in planets if p.owner != player]
    moves = []
    if not targets:
        return moves
    for mine in [p for p in planets if p.owner == player]:
        nearest = min(targets, key=lambda t: _math.hypot(mine.x - t.x, mine.y - t.y))
        need = max(int(nearest.ships) + 1, 20)
        if mine.ships >= need:
            ang = _math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, ang, need])
    return moves


arena = Arena({
    "v1_new": v1_new,
    "deepseek": deepseek_agent,    # 主要对照基线
    "v1_old": v1_old,              # 我们上一版
    "chatgpt": chatgpt_agent,      # 第三方对手
    "nearest": nearest_sniper,     # 官方 baseline
})

# 5 个 agent => C(5,2) = 10 个对子，每对 6 局，~6 分钟
print("=== Round Robin (6 episodes per pair, ~6 minutes) ===")
report = arena.round_robin(n_episodes=6)
arena.print_summary()


=== Round Robin (10 episodes per pair, ~5 minutes) ===
  v1_new vs v1_old: 

In [ ]:
# === Replay v1_new vs deepseek (visual inspection) ===
# 切换 OPPONENT 即可对比不同对手；P0 = 我方 (v1_new)
import importlib, sys
HERE = "/data1/huanghao3/KK/Orbit_War/scripts"
if HERE not in sys.path:
    sys.path.insert(0, HERE)
import v1.main as _v1_new
importlib.reload(_v1_new)
import v1_deepseek.main as _v1_deepseek
importlib.reload(_v1_deepseek)
import v1_chatgpt.main as _v1_chatgpt
importlib.reload(_v1_chatgpt)
import v1_old.main as _v1_old
importlib.reload(_v1_old)

OPPONENT_NAME = "deepseek"   # 改成 "chatgpt" / "v1_old" / "deepseek" 切换对手
opponents = {
    "deepseek": _v1_deepseek.agent,
    "chatgpt":  _v1_chatgpt.agent,
    "v1_old":   _v1_old.agent,
}

from kaggle_environments import make
env = make("orbit_wars", debug=True)
env.run([_v1_new.agent, opponents[OPPONENT_NAME]])

r0 = env.steps[-1][0]["reward"]
r1 = env.steps[-1][1]["reward"]
print(f"Player 0 (v1_new):  reward={r0}")
print(f"Player 1 ({OPPONENT_NAME}): reward={r1}")
env.render(mode="ipython", width=900, height=700)
